In [79]:
import os
import pandas as pd

In [80]:
df = pd.read_csv(r"D:\Crawl_Data\data\full_url_link.csv")
df.head()

,id,name,url_key,product_count,link,parent_id
0,9733,Kiến Thức Bách Khoa,kien-thuc-bach-khoa,10000.0,https://tiki.vn/kien-thuc-bach-khoa/c9733,873
1,871,Sách tư duy - Kỹ năng sống,sach-ky-nang-song,10000.0,https://tiki.vn/sach-ky-nang-song/c871,870
2,1084,"Truyện Tranh, Manga, Comic",truyen-tranh,8568.0,https://tiki.vn/truyen-tranh/c1084,316
3,853,Kiến thức - Bách khoa,kien-thuc-ky-nang,7773.0,https://tiki.vn/kien-thuc-ky-nang/c853,393
4,2248,Sách tham khảo cấp I,sach-tham-khao-cap-i,5331.0,https://tiki.vn/sach-tham-khao-cap-i/c2248,2320


In [81]:
# Số ID - dạng list
list_id = df["id"].copy().values
list_id[-10:]

array([67965, 67966, 67970, 67973, 67974, 67975, 67976, 67977, 68013,
       68014])

In [82]:
PRODUCTS = os.path.join("data", "products")
REVIEWS = os.path.join("data", "reviews")

PRODUCTS_LINK = lambda id : f"category_{id}.csv"
REVIEWS_LINK  = lambda id : f"category_{id}_reviews.csv"

## Kiểm tra đường dẫn và xem xét còn bao nhiêu file cần chia

In [83]:
miss_categories = []        # Thể loại chưa có sách
uncall_categories = []      # Thể loại x Sách chưa có bình luận

for id in list_id:
    if not os.path.isfile(os.path.join(PRODUCTS, PRODUCTS_LINK(id))):
        miss_categories.append(id)
    if not os.path.isfile(os.path.join(REVIEWS, REVIEWS_LINK(id))):
        uncall_categories.append(id)

print(f"Số thể loại thiếu sách: {len(miss_categories)}")
print(f"Số thể loại sách chưa cào bình luận: {len(uncall_categories)}")

Số thể loại thiếu sách: 1
Số thể loại sách chưa cào bình luận: 191


In [84]:
uncall_categories[:5]

[np.int64(9733),
 np.int64(871),
 np.int64(1084),
 np.int64(2248),
 np.int64(67979)]

In [85]:
lim = len(uncall_categories) // 3
KHANG = uncall_categories[:lim]
UYEN = uncall_categories[lim:-lim]
NGUYEN = uncall_categories[-lim:]

print(len(KHANG), len(UYEN), len(NGUYEN))

63 65 63


In [86]:

import os
import zipfile
import pandas as pd

# ==================================================
# 1. CẤU HÌNH
# ==================================================

BASE_DIR = os.getcwd()
PRODUCTS_DIR = os.path.join(BASE_DIR, "data", "products")

# Khai báo ID của từng thành viên
GROUPS = {
    "KHANG": KHANG,
    "UYEN": UYEN,
    "NGUYEN": NGUYEN,
}

# ==================================================
# 2. TẠO ZIP RIÊNG CHO TỪNG THÀNH VIÊN
# ==================================================

for name, ids in GROUPS.items():

    zip_path = os.path.join(
        BASE_DIR,
        f"{name.lower()}.zip"
    )

    missing_files = []
    added_files = 0

    with zipfile.ZipFile(
        zip_path, "w", zipfile.ZIP_DEFLATED
    ) as zf:

        # ------------------------------------------
        # A. Thêm các file category vào data/products
        # ------------------------------------------

        for category_id in ids:

            filename = f"category_{category_id}.csv"

            filepath = os.path.join(
                PRODUCTS_DIR,
                filename
            )

            if os.path.isfile(filepath):
                zf.write(
                    filepath,
                    arcname=f"data/products/{filename}"
                )
                added_files += 1
            else:
                missing_files.append(filename)

        # ------------------------------------------
        # B. Lọc full_url_link.csv theo ID
        # ------------------------------------------

        member_df = df[df["id"].isin(ids)]

        # Ghi CSV trực tiếp vào ZIP, không tạo file tạm
        csv_content = member_df.to_csv(
            index=False,
            encoding="utf-8-sig"
        )

        zf.writestr(
            "data/full_url_link.csv",
            csv_content
        )

    # ----------------------------------------------
    # C. Báo cáo
    # ----------------------------------------------

    print(f"\nThành viên: {name}")
    print(f"File ZIP: {zip_path}")
    print(f"Số file category đã thêm: {added_files}")
    print(f"Số dòng trong full_url_link.csv: {len(member_df)}")
    print(f"Số file category bị thiếu: {len(missing_files)}")

    if missing_files:
        print("Danh sách file thiếu:", missing_files)


Thành viên: KHANG
File ZIP: d:\Crawl_Data\khang.zip
Số file category đã thêm: 63
Số dòng trong full_url_link.csv: 63
Số file category bị thiếu: 0

Thành viên: UYEN
File ZIP: d:\Crawl_Data\uyen.zip
Số file category đã thêm: 65
Số dòng trong full_url_link.csv: 65
Số file category bị thiếu: 0

Thành viên: NGUYEN
File ZIP: d:\Crawl_Data\nguyen.zip
Số file category đã thêm: 62
Số dòng trong full_url_link.csv: 63
Số file category bị thiếu: 1
Danh sách file thiếu: ['category_67808.csv']
